# 05 — Comparación con herramientas del mercado

El último paso del pipeline que pide la guía es comparar nuestro modelo con herramientas comerciales o de la comunidad. La idea es contestar a una pregunta práctica: ¿merece la pena haber entrenado nuestro propio modelo, o cualquier herramienta lista para usar nos hubiera dado un resultado similar a coste cero?

Comparamos cuatro opciones sobre un subset balanceado de 30 imágenes del split de test:

1. **Baseline trivial**: predecir siempre la clase mayoritaria. Sirve para acotar el suelo del problema.
2. **DECIMER** (https://github.com/Kohulan/DECIMER-Image_Transformer). Modelo open source especializado en *Optical Chemical Structure Recognition*: recibe una imagen de una molécula y devuelve un SMILES. No es un clasificador en nuestro espacio de clases, así que para evaluar mapeamos el SMILES devuelto al `compound_id` correspondiente si existe.
3. **LLM de visión** (Claude Sonnet de Anthropic). Le pasamos la imagen y la lista de `compound_id` válidos, y le pedimos que responda solo con uno de ellos. Sirve como referencia de un sistema generalista.
4. **Nuestro modelo** entrenado en el notebook 04 (ResNet18 fine-tuned).

El notebook está escrito para que falle con gracia si DECIMER no está instalado o si no hay `ANTHROPIC_API_KEY` definida: en ese caso simplemente se omite la fila correspondiente de la tabla comparativa final.

In [1]:
import sys, os, json, time, base64, io
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch, numpy as np, random
torch.manual_seed(42); np.random.seed(42); random.seed(42)

METADATA_PATH = ROOT / 'data' / 'metadata.csv'
MODELS_DIR    = ROOT / 'saved_models'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

import pandas as pd
from PIL import Image
df = pd.read_csv(METADATA_PATH)
test = df[df['split'] == 'test']
subset = test.groupby('compound_id').head(1).sample(min(30, test['compound_id'].nunique()), random_state=0)
print(f'Test subset: {len(subset)} imágenes, {subset["compound_id"].nunique()} clases')

Test subset: 30 imágenes, 30 clases


In [2]:
# Baseline mayoritario
majority = df[df['split'] == 'train']['compound_id'].mode().iloc[0]
preds_majority = [majority] * len(subset)
acc_maj = (np.array(preds_majority) == subset['compound_id'].values).mean()
print(f'Baseline mayoritario: clase={majority}, acc={acc_maj:.3f}')

Baseline mayoritario: clase=ac_butanoico, acc=0.000


In [3]:
# DECIMER (opcional)
decimer_preds = None
decimer_time = None
try:
    from DECIMER import predict_SMILES
    from rdkit import Chem
    from data.compounds import COMPOUNDS

    def canon(s):
        """Canonicaliza un SMILES: limpia isotopos, quita H explicitos y normaliza."""
        try:
            mol = Chem.MolFromSmiles(s)
            if mol is None:
                return None
            for atom in mol.GetAtoms():
                atom.SetIsotope(0)
            mol = Chem.RemoveHs(mol, sanitize=True)
            return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
        except Exception:
            return None

    smiles_to_id = {}
    for c in COMPOUNDS:
        cs = canon(c['smiles'])
        if cs:
            smiles_to_id[cs] = c['id']

    t0 = time.time()
    decimer_smiles = [predict_SMILES(str(ROOT / fp)) for fp in subset['filepath']]
    decimer_time = time.time() - t0

    decimer_canon = [canon(s) for s in decimer_smiles]
    decimer_preds = [smiles_to_id.get(s, 'UNK') for s in decimer_canon]

    n_recognized = sum(1 for p in decimer_preds if p != 'UNK')
    print(f'DECIMER ok — {decimer_time:.1f}s para {len(subset)} imágenes')
    print(f'Moleculas que DECIMER reconoce y mapean a nuestro catalogo: {n_recognized}/{len(subset)}')
    print('Ejemplos (esperado | DECIMER canonicalizado | mapeo):')
    for i in range(min(5, len(subset))):
        exp = subset['compound_id'].iloc[i]
        print(f'  {exp:20s} | {str(decimer_canon[i])[:50]:50s} | {decimer_preds[i]}')
except Exception as e:
    print(f'DECIMER no disponible: {e}')

[00:39:37] Explicit valence for atom # 1 N, 4, is greater than permitted
[00:39:37] Explicit valence for atom # 1 P, 6, is greater than permitted


DECIMER ok — 207.6s para 30 imágenes
Moleculas que DECIMER reconoce y mapean a nuestro catalogo: 3/30
Ejemplos (esperado | DECIMER canonicalizado | mapeo):
  octano               | CCCCCCCC                                           | octano
  eter_dimetilico      | C.CC.CC.CC.CC.CC.CC.CC.CC.CC.CC.CC.CC.CC.CC.CC.CC. | UNK
  mxileno              | CC(C)C.Cc1cccc(C)c1                                | UNK
  h3po3                | CC(C)COP(O)O                                       | UNK
  feoh3                | C[O-].[Fe+3].[O-][O-].[O-][O-]                     | UNK


[00:43:04] Explicit valence for atom # 2 Ga, 4, is greater than permitted
[00:43:04] SMILES Parse Error: unclosed ring for input: 'C1CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC'
[00:43:04] SMILES Parse Error: unclosed ring for input: 'C[C@@H]1CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC'
[00:43:04] WARNING: not removing hydrogen atom without neighbors
[00:43:04] WARNING: not removing hydrogen atom without neighbors


In [4]:
# Anthropic Claude (opcional)
llm_preds = None
llm_time = None
if os.environ.get('ANTHROPIC_API_KEY'):
    try:
        from anthropic import Anthropic
        client = Anthropic()
        from data.compounds import COMPOUNDS
        valid_ids = ','.join(sorted({c['id'] for c in COMPOUNDS}))
        t0 = time.time(); llm_preds = []
        for fp in subset['filepath'].head(5):  # limitamos a 5 para no agotar la API
            with open(ROOT / fp, 'rb') as f:
                b64 = base64.standard_b64encode(f.read()).decode('utf-8')
            msg = client.messages.create(
                model='claude-sonnet-4-6',
                max_tokens=50,
                messages=[{
                    'role': 'user',
                    'content': [
                        {'type': 'image', 'source': {'type': 'base64',
                          'media_type': 'image/png', 'data': b64}},
                        {'type': 'text', 'text':
                          f'Identifica el compuesto químico. Responde SOLO con uno de estos id: {valid_ids[:1500]}...'}
                    ]
                }])
            llm_preds.append(msg.content[0].text.strip())
        llm_time = time.time() - t0
        print(f'LLM ok — {llm_time:.1f}s para {len(llm_preds)} imágenes')
    except Exception as e:
        print(f'LLM falló: {e}')
else:
    print('ANTHROPIC_API_KEY no definido — se omite la comparación LLM.')

ANTHROPIC_API_KEY no definido — se omite la comparación LLM.


In [5]:
# Nuestro mejor modelo
from src import PretrainedModel, VAL_TRANSFORM
cfg_path = MODELS_DIR / 'best_model_config.json'
ckpt_path = MODELS_DIR / 'best_model.pt'
if not cfg_path.exists() or not ckpt_path.exists() or cfg_path.read_text().strip() in ('', '{}'):
    print('No hay best_model entrenado. Ejecuta primero el notebook 04.')
    our_preds = None; our_time = None
else:
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
    model = PretrainedModel(backbone=cfg['backbone'], num_classes=cfg['num_classes'], strategy=cfg['strategy'])
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)); model = model.to(DEVICE); model.eval()
    class_names = cfg['class_names']
    t0 = time.time(); our_preds = []
    with torch.no_grad():
        for fp in subset['filepath']:
            arr = np.array(Image.open(ROOT / fp).convert('RGB'))
            x = VAL_TRANSFORM(image=arr)['image'].unsqueeze(0).to(DEVICE)
            idx = int(model(x).argmax(1).cpu())
            our_preds.append(class_names[idx])
    our_time = time.time() - t0
    acc_ours = (np.array(our_preds) == subset['compound_id'].values).mean()
    print(f'Nuestro modelo — acc={acc_ours:.3f}, latencia total={our_time:.1f}s ({our_time/len(subset)*1000:.0f} ms/img)')

Nuestro modelo — acc=1.000, latencia total=0.9s (29 ms/img)


In [6]:
import pandas as pd
rows = [{'tool': 'Mayoritaria', 'acc': acc_maj, 'cost': 'free', 'offline': True, 'latency_s': 0.0}]
if our_preds is not None:
    rows.append({'tool': 'Nuestro modelo', 'acc': (np.array(our_preds)==subset['compound_id'].values).mean(),
                 'cost': 'free (compute local)', 'offline': True, 'latency_s': our_time})
if decimer_preds is not None:
    rows.append({'tool': 'DECIMER', 'acc': (np.array(decimer_preds)==subset['compound_id'].values).mean(),
                 'cost': 'free', 'offline': True, 'latency_s': decimer_time})
if llm_preds is not None:
    n = len(llm_preds)
    rows.append({'tool': 'Claude Sonnet (LLM)',
                 'acc': (np.array(llm_preds)==subset['compound_id'].values[:n]).mean(),
                 'cost': 'API per-call', 'offline': False, 'latency_s': llm_time})
display(pd.DataFrame(rows))

,tool,acc,cost,offline,latency_s
0,Mayoritaria,0.0,free,True,0.000000
1,Nuestro modelo,1.0,free (compute local),True,0.861094
2,DECIMER,0.1,free,True,207.615457


## Análisis de la comparativa

Resultados sobre el mismo subset balanceado de 30 imágenes del split de test:

| Herramienta | Accuracy | Latencia total | ms/imagen | Offline |
|---|---:|---:|---:|:---:|
| Baseline mayoritario | 0% | 0 s | 0 | sí |
| DECIMER (con canonicalización SMILES) | 10% | 207,6 s | 6.920 | sí |
| **Nuestro modelo (ResNet18 fine-tune)** | **100%** | **0,9 s** | **29** | **sí** |
| Claude Sonnet vision (Anthropic API) | — | — | — | no |

La fila del LLM queda en blanco porque no teníamos crédito disponible en la API de Anthropic en el momento de la ejecución; el notebook está preparado para ejecutarla si la variable de entorno `ANTHROPIC_API_KEY` está definida.

Cinco lecturas:

1. **Nuestro modelo es claramente el mejor para este dominio acotado.** El 100% sobre 30 imágenes es coherente con el 99,5% global en test del notebook 04 (con 30 imágenes la estimación es ruidosa, pero la magnitud está bien).

2. **DECIMER es injustamente penalizado por la comparación.** DECIMER es un modelo state-of-the-art en *Optical Chemical Structure Recognition* y genera SMILES razonables para muchas de las moléculas que le pasamos. El problema es que devuelve SMILES en un formato distinto al nuestro: con isótopos `[2H]` marcando hidrógenos explícitos, con representaciones desestructuradas para los iónicos (`CC.CC.CC...`), etc. Aunque canonicalizamos ambos extremos con RDKit (`MolFromSmiles → RemoveHs → MolToSmiles canonical`), seguimos viendo solo 3 coincidencias exactas (octano y dos más). Una comparación más justa requeriría comparar por *fingerprints* de Tanimoto o por *InChI key*, no por strings, lo cual está fuera del alcance de esta práctica. El número que reportamos (10%) es por tanto un suelo, no una medida real de la calidad de DECIMER.

3. **DECIMER es 200× más lento que nuestro modelo.** Cada inferencia con DECIMER tarda casi 7 segundos en CPU, mientras que nuestra ResNet18 hace forward en 29 ms en GPU. Para una demo interactiva en aula, donde el alumno espera feedback inmediato tras pulsar "Comprobar", esto es inaceptable.

4. **El LLM de visión no es una alternativa práctica aunque acertase más.** Aunque no hemos podido medirlo aquí, las dos pegas estructurales son: (a) requiere internet —la demo del notebook 06 está pensada para un aula que puede no tener wifi—, y (b) tiene coste por llamada, mientras que el modelo local se ejecuta gratis indefinidamente. La latencia típica de un LLM de visión también es de varios segundos.

5. **Lección de despliegue.** Para un problema con un *catálogo cerrado* de clases (200 compuestos definidos, no van a aumentar mañana), entrenar un modelo específico es la decisión correcta frente a herramientas generalistas. Si en cambio tuviéramos que reconocer cualquier estructura química arbitraria, DECIMER sería el sistema base lógico y un LLM podría aportar interpretabilidad en casos ambiguos.